#### Импорт

In [12]:
import os
import sys
import math
import itertools
from pathlib import Path

# Добавляем путь на уровень выше
sys.path.append(str(Path(os.getcwd()).resolve().parent))
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


from utils.features import *
from utils.load_data import load_all_data

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary
from tqdm import tqdm


#### Загрузка данных

In [13]:
data_dir = Path('../data/METR-LA')
metadata, data, adj = load_all_data(data_dir)
data = data[:2016]#data[288*2:2016+288+288]
data = data.copy()

data[:, :, 1] = data[:, :, 1] * 288
data[:, :, 2] = data[:, :, 2] * 7

print(f"data.shape: {data.shape}")


data.shape: (2016, 207, 3)


#### DataLoader

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
adj_tensor = torch.tensor(adj, dtype=torch.float32).to(device)
print(f"Using device: {device}")

# Данные и параметры
L, N, C = data.shape  # [2016, N, C]
batch_size = 16
train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2
seq_len = 12  # Количество временных шагов на вход
pred_len = 12  # Количество временных шагов для предсказания
num_lags = 0

# Индексы каналов для нормализации


# Проверка корректности разделения данных
assert train_ratio + val_ratio + test_ratio == 1.0, "Сумма долей train, val и test должна быть равна 1.0"

# Разделение данных на train, val и test
num_samples = data.shape[0]  # Количество временных шагов (L)
train_size = int(num_samples * train_ratio)
val_size = int(num_samples * val_ratio)
test_size = num_samples - train_size - val_size

train_data = data[:train_size, :, :]  # [train_size, N, C]
val_data = data[train_size:train_size + val_size, :, :]  # [val_size, N, C]
test_data = data[train_size + val_size:, :, :]  # [test_size, N, C]

def normalize_data(train_data, val_data, test_data, channels_to_normalize, eps=1e-5):
    assert all(0 <= ch < train_data.shape[2] for ch in channels_to_normalize), "Недопустимые индексы каналов"

    # Вычисляем среднее и стандартное отклонение по обучающим данным
    mean = train_data[:, :, channels_to_normalize].mean(axis=(0, 1), keepdims=True)  # [1, 1, C_norm]
    std = train_data[:, :, channels_to_normalize].std(axis=(0, 1), keepdims=True)    # [1, 1, C_norm]
    std[std < eps] = 1.0  # защита от деления на 0

    # Нормализация
    train_data[:, :, channels_to_normalize] = (train_data[:, :, channels_to_normalize] - mean) / std
    val_data[:, :, channels_to_normalize] = (val_data[:, :, channels_to_normalize] - mean) / std
    test_data[:, :, channels_to_normalize] = (test_data[:, :, channels_to_normalize] - mean) / std

    return train_data, val_data, test_data, mean, std

# Нормализация данных
normalize = True
if normalize:
    channels_to_normalize = [0]  # например, только поток
    train_data, val_data, test_data, mean, std = normalize_data(train_data, val_data, test_data, channels_to_normalize)


# Создание кастомного Dataset
class TrafficDataset(Dataset):
    def __init__(self, data, seq_len, pred_len, num_lags):
        """
        Параметры:
          data: тензор формы [L, N, C]
          seq_len: длина входной последовательности
          pred_len: длина предсказываемой последовательности
          num_lags: число лагов, добавляемых к признакам (из канала C=0)
        """
        super().__init__()
        self.data = data  # [L, N, C]
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_lags = num_lags
        

    def __len__(self):
        # Количество возможных последовательностей
        return self.data.shape[0] - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        # Извлекаем основную последовательность входных данных
        x_main = self.data[idx: idx + self.seq_len, :, :]  # [seq_len, N, C]

        # Формируем матрицу лагов для канала C=0 внутри текущей последовательности.
        # Если для временного шага t нет достаточного количества предыдущих значений,
        # соответствующие позиции заполняются нулями.
        lag_matrix = torch.zeros((self.seq_len, self.data.shape[1], self.num_lags), device=self.data.device)
        for t in range(self.seq_len):
            # Для каждого лага, lag=1 соответствует непосредственному предыдущему значению
            for lag in range(1, self.num_lags + 1):
                if t - lag >= 0:
                    lag_matrix[t, :, lag - 1] = x_main[t - lag, :, 0]
                # Если t - lag < 0, оставляем нули

        # Объединяем исходные признаки и лаги по последнему измерению
        x = torch.cat([x_main, lag_matrix], dim=-1)  # [seq_len, N, C + num_lags]

        # Целевые значения – поток (канал C=0) для последовательности предсказания
        y = self.data[idx + self.seq_len: idx + self.seq_len + self.pred_len, :, 0]  # [pred_len, N]

        return x, y

# Преобразование данных в тензоры с dtype=torch.float32
train_data = torch.tensor(train_data, dtype=torch.float32).to(device)
val_data = torch.tensor(val_data, dtype=torch.float32).to(device)
test_data = torch.tensor(test_data, dtype=torch.float32).to(device)

# Создание DataLoader
train_dataset = TrafficDataset(train_data, seq_len, pred_len, num_lags)
val_dataset = TrafficDataset(val_data, seq_len, pred_len, num_lags)
test_dataset = TrafficDataset(test_data, seq_len, pred_len, num_lags)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Using device: cuda


In [15]:
for x, y in train_loader:
    print(f"Shape of x (input data):    {x.shape}")  # [B, L, N, C]
    print(f"Shape of y (target data):   {y.shape}")  # [B, L, N]

    # Проверка первых двух сенсоров и временных шагов
    print("\nFirst two sensors and time steps in x:")
    print(x[0, :2, :2, :3])  # Первый батч, первые два временных шага, первые два сенсора, все каналы

    print("\nFirst two sensors and time steps in y:")
    print(y[0, :2, :2])  # Первый батч, первые два временных шага, первые два сенсора

    print("\nExample lags data:")
    print(x[0, :, 0, :])

    # Проверка типов данных для всех каналов
    print("\nData types for each channel in x:")
    for channel in range(x.shape[3]):  # Проходим по всем каналам
        print(f"Channel {channel} dtype: {x[0, 0, 0, channel].dtype}, device: {x[0, 0, 0, channel].device}")

    # Остановка выполнения для ручной проверки
    break

Shape of x (input data):    torch.Size([16, 12, 207, 3])
Shape of y (target data):   torch.Size([16, 12, 207])

First two sensors and time steps in x:
tensor([[[  0.4000, 122.0000,   4.0000],
         [  0.4545, 122.0000,   4.0000]],

        [[  0.3915, 123.0000,   4.0000],
         [  0.6138, 123.0000,   4.0000]]], device='cuda:0')

First two sensors and time steps in y:
tensor([[0.5065, 0.4605],
        [0.3991, 0.5065]], device='cuda:0')

Example lags data:
tensor([[  0.4000, 122.0000,   4.0000],
        [  0.3915, 123.0000,   4.0000],
        [  0.3114, 124.0000,   4.0000],
        [  0.4451, 125.0000,   4.0000],
        [  0.4145, 126.0000,   4.0000],
        [  0.5226, 127.0000,   4.0000],
        [  0.5218, 128.0000,   4.0000],
        [  0.4613, 129.0000,   4.0000],
        [  0.5984, 130.0000,   4.0000],
        [  0.5226, 131.0000,   4.0000],
        [  0.3302, 132.0000,   4.0000],
        [  0.4758, 133.0000,   4.0000]], device='cuda:0')

Data types for each channel in x:
C

#### Adapter

In [16]:
class LearnableFilter(nn.Module):
    def __init__(
        self,
        num_features,
        seq_len,
        init_range=0.002,
        freq_mask_threshold=None  # например, оставить первые 20 частот
    ):
        """
        num_features: число каналов для фильтрации
        seq_len: длина входной последовательности (L)
        init_range: диапазон инициализации весов фильтра (для стабильности)
        freq_mask_threshold: если не None, используем только фильтры по низким частотам (например, 20)
        """
        super().__init__()
        self.seq_len = seq_len
        self.num_features = num_features
        # Размерность FFT-части (только положительные частоты)
        self.fft_len_half = seq_len // 2 + 1

        real_part = torch.ones(1, num_features, 1, self.fft_len_half)
        imag_part = torch.zeros(1, num_features, 1, self.fft_len_half)
        nn.init.uniform_(real_part, -init_range, init_range)
        nn.init.uniform_(imag_part, -init_range, init_range)

        self.K_real = nn.Parameter(real_part)
        self.K_imag = nn.Parameter(imag_part)

        if freq_mask_threshold is not None:
            assert freq_mask_threshold <= self.fft_len_half, "Порог частотного фильтра больше, чем половина длины последовательности"
            self.freq_mask_threshold = freq_mask_threshold
        else:
            self.freq_mask_threshold = 2

    def forward(self, x):
        """
        x: (B, Channels, N, L)
        """
        batch, channels, nodes, seq_len_in = x.shape
        assert seq_len_in == self.seq_len, f"Входной seq_len={seq_len_in}, ожидалось {self.seq_len}"

        x_fft = torch.fft.rfft(x, n=self.seq_len, dim=-1)  # (B, Channels, N, fft_len_half)

        # (1) Маска частот, если задан порог (например, freq_mask_threshold = 20)
        K_real = self.K_real
        K_imag = self.K_imag
        if self.freq_mask_threshold is not None:
            mask = torch.zeros_like(K_real)
            mask[..., :self.freq_mask_threshold] = 1.0
            K_real = K_real * mask
            K_imag = K_imag * mask

        K = torch.complex(K_real, K_imag).to(x.device)  # Преобразуем в комплексное число

        # (2) Применение фильтра
        x_filtered_fft = x_fft * K  # Broadcasting

        # (3) Обратное FFT
        x_filtered = torch.fft.irfft(x_filtered_fft, n=self.seq_len, dim=-1)
        if torch.isnan(x_filtered).any():
            raise RuntimeError("NaN найден в результате LearnableFilter")
        return x_filtered * 0.1
    
    def reg_loss(self, l2_coeff=1e-4, smooth_coeff=1e-4):
        """
        l2_coeff: коэффициент L2-регуляризации
        smooth_coeff: коэффициент гладкости (разность по частотам)
        """
        l2 = (self.K_real ** 2).sum() + (self.K_imag ** 2).sum()
        # smoothness loss: (f_k - f_{k-1})^2 для всех k > 0
        smooth_real = ((self.K_real[..., 1:] - self.K_real[..., :-1]) ** 2).sum()
        smooth_imag = ((self.K_imag[..., 1:] - self.K_imag[..., :-1]) ** 2).sum()
        smooth = smooth_real + smooth_imag
        return l2_coeff * l2 + smooth_coeff * smooth
    

class TimeEmbedding(nn.Module):
    # 2. TimeEmbedding: получение эмбеддингов времени с дополнением последовательности при необходимости
    def __init__(self, num_embeddings=288, emb_dim=10):
        """
        Args:
            num_embeddings: количество уникальных временных индексов.
            emb_dim: размерность эмбеддингов времени.
        """
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, emb_dim)

    def forward(self, time_idx: torch.Tensor):
        """
        Args:
            time_idx: тензор индексов времени формы (B, N, L).
            target_seq_len: требуемая длина последовательности.
        Returns:
            Эмбеддинги времени формы (B, emb_dim, N, L)
        """
        time_emb = self.embedding(time_idx)
        # Перестановка размерностей для конкатенации: (B, N, L, emb_dim) -> (B, emb_dim, N, L)
        time_emb = time_emb.permute(0, 3, 1, 2)
        return time_emb

class UniversalInputAdapter(nn.Module):
    def __init__(
        self,
        seq_len,
        n_nodes,
        residual_channels,
        use_filter=False,
        use_time_emb=False,
        use_nodes=False,
        use_gate=False,
        dropout_rate=0.0,
    ):
        super().__init__()

        self.use_filter = use_filter
        self.use_time_emb = use_time_emb
        self.use_nodes = use_nodes
        self.use_gate = use_gate
        self.dropout_rate = dropout_rate

        extra_channels = 0

        if self.use_time_emb:
            self.time_embedding = TimeEmbedding(num_embeddings=288, emb_dim=10)
            self.dropout_time = nn.Dropout2d(dropout_rate)
            extra_channels += 10

        if self.use_nodes:
            self.node_embedding = nn.Embedding(n_nodes, 10)
            self.dropout_node = nn.Dropout2d(dropout_rate)
            extra_channels += 10

        if self.use_filter:
            self.filter = LearnableFilter(num_features=1, seq_len=seq_len, init_range=0.002)
        else:
            self.filter = None

        self.start_conv = nn.Conv2d(
            in_channels=1 + extra_channels,
            out_channels=residual_channels,
            kernel_size=(1, 1)
        )

        if self.use_gate:
            self.gate_conv = nn.Conv2d(1 + extra_channels, 1 + extra_channels, kernel_size=(1, 1))


    def forward(self, history_data):
        """
        history_data: (B, N, L, C) — входной тензор
        """
        x = history_data.permute(0, 3, 1, 2).contiguous()  # (B, C, N, L)
        speed = x[:, [0], :, :]  # (B, 1, N, L)
        time = x[:, 1, :, :]     # (B, N, L)

        if self.use_filter:
            speed = speed + self.filter(speed.transpose(2, 3)).transpose(2, 3)

        components = [speed]

        if self.use_time_emb:
            time_emb = self.time_embedding(time.long())
            time_emb = self.dropout_time(time_emb)
            components.append(time_emb)

        if self.use_nodes:
            B, _, N, T = speed.shape
            nodes = self.node_embedding(torch.arange(N, device=x.device))  # [N, 10]
            nodes = nodes.unsqueeze(0).unsqueeze(-1).expand(B, -1, -1, T)  # [B, N, 10, T]
            nodes = nodes.permute(0, 2, 1, 3)  # [B, 10, N, T]
            nodes = self.dropout_node(nodes)
            components.append(nodes)

        x_combined = torch.cat(components, dim=1)

        if self.use_gate:
            gate = torch.sigmoid(self.gate_conv(x_combined))
            x_combined = x_combined * gate

        x_out = self.start_conv(x_combined)
        return x_out

In [17]:
class LearnableFilter(nn.Module):
    def __init__(
        self,
        num_features,
        seq_len,
        init_range=0.002,
        freq_mask_threshold=None  # например, оставить первые 20 частот
    ):
        """
        num_features: число каналов для фильтрации
        seq_len: длина входной последовательности (L)
        init_range: диапазон инициализации весов фильтра (для стабильности)
        freq_mask_threshold: если не None, используем только фильтры по низким частотам (например, 20)
        """
        super().__init__()
        self.seq_len = seq_len
        self.num_features = num_features
        # Размерность FFT-части (только положительные частоты)
        self.fft_len_half = seq_len // 2 + 1

        real_part = torch.ones(1, num_features, 1, self.fft_len_half)
        imag_part = torch.zeros(1, num_features, 1, self.fft_len_half)
        nn.init.uniform_(real_part, -init_range, init_range)
        nn.init.uniform_(imag_part, -init_range, init_range)

        self.K_real = nn.Parameter(real_part)
        self.K_imag = nn.Parameter(imag_part)

        if freq_mask_threshold is not None:
            assert freq_mask_threshold <= self.fft_len_half, "Порог частотного фильтра больше, чем половина длины последовательности"
            self.freq_mask_threshold = freq_mask_threshold
        else:
            self.freq_mask_threshold = self.fft_len_half

    def forward(self, x):
        """
        x: (B, Channels, N, L)
        """
        batch, channels, nodes, seq_len_in = x.shape
        assert seq_len_in == self.seq_len, f"Входной seq_len={seq_len_in}, ожидалось {self.seq_len}"

        x_fft = torch.fft.rfft(x, n=self.seq_len, dim=-1)  # (B, Channels, N, fft_len_half)

        # (1) Маска частот, если задан порог (например, freq_mask_threshold = 20)
        K_real = self.K_real
        K_imag = self.K_imag
        if self.freq_mask_threshold < self.fft_len_half:
            mask = torch.zeros_like(K_real)
            mask[..., :self.freq_mask_threshold] = 1.0
            K_real = K_real * mask
            K_imag = K_imag * mask

        K = torch.complex(K_real, K_imag).to(x.device)  # Преобразуем в комплексное число

        # (2) Применение фильтра
        x_filtered_fft = x_fft * K  # Broadcasting

        # (3) Обратное FFT
        x_filtered = torch.fft.irfft(x_filtered_fft, n=self.seq_len, dim=-1)
        if torch.isnan(x_filtered).any():
            raise RuntimeError("NaN найден в результате LearnableFilter")
        return x_filtered * 0.1
    
    def reg_loss(self, l2_coeff=1e-4, smooth_coeff=1e-4):
        """
        l2_coeff: коэффициент L2-регуляризации
        smooth_coeff: коэффициент гладкости (разность по частотам)
        """
        l2 = (self.K_real ** 2).sum() + (self.K_imag ** 2).sum()
        # smoothness loss: (f_k - f_{k-1})^2 для всех k > 0
        smooth_real = ((self.K_real[..., 1:] - self.K_real[..., :-1]) ** 2).sum()
        smooth_imag = ((self.K_imag[..., 1:] - self.K_imag[..., :-1]) ** 2).sum()
        smooth = smooth_real + smooth_imag
        return l2_coeff * l2 + smooth_coeff * smooth


class TimeEmbedding(nn.Module):
    def __init__(self, num_embeddings=288, emb_dim=10):
        """
        Args:
            num_embeddings: количество уникальных временных индексов.
            emb_dim: размерность эмбеддингов времени.
        """
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, emb_dim)

    def forward(self, time_idx: torch.Tensor):
        """
        Args:
            time_idx: тензор индексов времени формы (B, N, L).
        Returns:
            Эмбеддинги времени формы (B, emb_dim, N, L)
        """
        time_emb = self.embedding(time_idx)
        # Перестановка размерностей для конкатенации: (B, N, L, emb_dim) -> (B, emb_dim, N, L)
        time_emb = time_emb.permute(0, 3, 1, 2)
        return time_emb


# Базовый класс для всех механизмов управления
class BaseGate(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        return x


# Компонентно-специфические механизмы управления
class ComponentAttentionGate(BaseGate):
    def __init__(self, in_channels, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, max(in_channels // reduction, 1), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(in_channels // reduction, 1), in_channels, bias=False),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        b, c, n, t = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class SpatioTemporalGate(BaseGate):
    def __init__(self, in_channels):
        super().__init__()
        # Пространственное управление (по узлам)
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(in_channels, 1, kernel_size=(1, 1)),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        # Временное управление (по времени)
        self.temporal_gate = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=(1, 1)),
            nn.BatchNorm2d(in_channels),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        # Применяем пространственное внимание
        spatial_att = self.spatial_gate(x)
        x_spatial = x * spatial_att
        # Применяем временное внимание
        temporal_att = self.temporal_gate(x_spatial)
        return x_spatial * temporal_att


class FrequencyAwareGate(BaseGate):
    def __init__(self, in_channels, seq_len):
        super().__init__()
        self.seq_len = seq_len
        self.fft_len = seq_len // 2 + 1
        
        # Механизм управления в частотной области
        self.freq_gate = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=(1, 1)),
            nn.BatchNorm2d(in_channels),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        b, c, n, t = x.size()
        
        # Преобразование в частотную область
        x_t = x.transpose(2, 3)  # [B, C, T, N]
        x_fft = torch.fft.rfft(x_t, dim=2, norm="ortho")
        
        # Разделение на амплитуду и фазу
        magnitude = torch.abs(x_fft)
        phase = torch.angle(x_fft)
        
        # Объединение информации о амплитуде и фазе
        freq_features = torch.cat([
            magnitude.transpose(2, 3),  # [B, C, N, fft_len]
            phase.transpose(2, 3)       # [B, C, N, fft_len]
        ], dim=1)
        
        # Применение управления в частотной области
        freq_gate = self.freq_gate(freq_features)
        
        # Привести к нужному размеру для временной области
        freq_gate = F.interpolate(
            freq_gate, size=(n, t), mode='bilinear', align_corners=False
        )
        
        return x * freq_gate


class MultiResolutionGate(BaseGate):
    def __init__(self, in_channels):
        super().__init__()
        self.gates = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, in_channels, kernel_size=(1, scale), padding=(0, scale//2)),
                nn.BatchNorm2d(in_channels),
                nn.Sigmoid()
            ) for scale in [1, 3, 5, 7, 9, 11]  # Несколько временных разрешений
        ])
        
    def forward(self, x):
        gates = [gate(x) for gate in self.gates]
        # Объединение всех механизмов управления (произведение всех gates)
        combined_gate = gates[0]
        for gate in gates[1:]:
            combined_gate = combined_gate * gate
        return x * combined_gate


class NodeAwareGate(BaseGate):
    def __init__(self, in_channels, num_nodes):
        super().__init__()
        self.node_gate = nn.Parameter(torch.ones(1, 1, num_nodes, 1))
        self.channel_gate = nn.Conv2d(in_channels, in_channels, kernel_size=(1, 1))
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # Учиться определять важность каждого узла
        node_importance = self.sigmoid(self.node_gate)
        node_importance = node_importance.expand_as(x)  # Расширение до формы x

        # Учиться определять важность каждого канала
        channel_importance = self.sigmoid(self.channel_gate(x))
        # Объединение
        return x * node_importance * channel_importance


# Функция для создания механизма управления по типу
def create_gate(gate_type, channels, seq_len=None, num_nodes=None):
    if gate_type == 'component':
        return ComponentAttentionGate(channels)
    elif gate_type == 'spatiotemporal':
        return SpatioTemporalGate(channels)
    elif gate_type == 'frequency':
        assert seq_len is not None, "Для FrequencyAwareGate требуется seq_len"
        return FrequencyAwareGate(channels, seq_len)
    elif gate_type == 'multiresolution':
        return MultiResolutionGate(channels)
    elif gate_type == 'node':
        assert num_nodes is not None, "Для NodeAwareGate требуется num_nodes"
        return NodeAwareGate(channels, num_nodes)
    else:
        return BaseGate()  # Пустой gate (без изменений)


class UniversalInputAdapter(nn.Module):
    def __init__(
        self,
        seq_len,
        n_nodes,
        residual_channels,
        # Основные флаги компонентов
        use_filter=False,
        use_time_emb=False,
        use_nodes=False,
        # Флаги для механизмов управления каждого компонента
        use_filter_gate=False,
        use_time_gate=False,
        use_node_gate=False,
        use_combined_gate=False,
        # Типы механизмов управления
        filter_gate_type='frequency',
        time_gate_type='spatiotemporal',
        node_gate_type='node',
        combined_gate_type='multiresolution',
        # Дополнительные параметры
        dropout_rate=0.0,
        freq_mask_threshold=None,
        time_emb_dim=10,
        node_emb_dim=10
    ):
        super().__init__()

        # Сохраняем флаги
        self.use_filter = use_filter
        self.use_time_emb = use_time_emb
        self.use_nodes = use_nodes
        
        self.use_filter_gate = use_filter_gate
        self.use_time_gate = use_time_gate
        self.use_node_gate = use_node_gate
        self.use_combined_gate = use_combined_gate
        
        self.dropout_rate = dropout_rate
        self.seq_len = seq_len
        self.n_nodes = n_nodes

        # Счетчики каналов
        speed_channels = 1
        time_channels = time_emb_dim if use_time_emb else 0
        node_channels = node_emb_dim if use_nodes else 0
        total_channels = speed_channels + time_channels + node_channels

        # 1. Фильтр скорости
        if self.use_filter:
            self.filter = LearnableFilter(
                num_features=1, 
                seq_len=seq_len, 
                init_range=0.002,
                freq_mask_threshold=freq_mask_threshold
            )
            # Gate для фильтра
            if self.use_filter_gate:
                self.filter_gate = create_gate(
                    filter_gate_type, 1, seq_len, n_nodes
                )
        else:
            self.filter = None
            self.filter_gate = None

        # 2. Эмбеддинги времени
        if self.use_time_emb:
            self.time_embedding = TimeEmbedding(num_embeddings=288, emb_dim=time_emb_dim)
            self.dropout_time = nn.Dropout2d(dropout_rate)
            # Gate для временных эмбеддингов
            if self.use_time_gate:
                self.time_gate = create_gate(
                    time_gate_type, time_emb_dim, seq_len, n_nodes
                )
        else:
            self.time_embedding = None
            self.dropout_time = None
            self.time_gate = None

        # 3. Эмбеддинги узлов
        if self.use_nodes:
            self.node_embedding = nn.Embedding(n_nodes, node_emb_dim)
            self.dropout_node = nn.Dropout2d(dropout_rate)
            # Gate для узловых эмбеддингов
            if self.use_node_gate:
                self.node_gate = create_gate(
                    node_gate_type, node_emb_dim, seq_len, n_nodes
                )
        else:
            self.node_embedding = None
            self.dropout_node = None
            self.node_gate = None

        # 4. Gate для объединенных данных
        if self.use_combined_gate:
            self.combined_gate = create_gate(
                combined_gate_type, total_channels, seq_len, n_nodes
            )
        else:
            self.combined_gate = None

        # 5. Выходной слой
        self.start_conv = nn.Conv2d(
            in_channels=total_channels,
            out_channels=residual_channels,
            kernel_size=(1, 1)
        )

    def forward(self, history_data):
        """
        history_data: (B, L, N, C) — входной тензор
        """
        x = history_data.permute(0, 3, 2, 1).contiguous()  # (B, C, N, L)
        
        # 1. Обработка скорости
        speed = x[:, [0], :, :]  # (B, 1, N, L)
        time = x[:, 1, :, :] if x.size(1) > 1 else None  # (B, N, L) or None
        
        if self.use_filter:
            filtered = self.filter(speed)
            if self.use_filter_gate:
                filtered = self.filter_gate(filtered)
            speed = speed + filtered

        components = [speed]

        # 2. Обработка временных эмбеддингов
        if self.use_time_emb and time is not None:
            time_emb = self.time_embedding(time.long())
            time_emb = self.dropout_time(time_emb)
            if self.use_time_gate:
                time_emb = self.time_gate(time_emb)
            components.append(time_emb)

        # 3. Обработка узловых эмбеддингов
        if self.use_nodes:
            B, _, N, T = speed.shape
            nodes = self.node_embedding(torch.arange(N, device=x.device))  # [N, emb_dim]
            nodes = nodes.unsqueeze(0).unsqueeze(-1).expand(B, -1, -1, T)  # [B, N, emb_dim, T]
            nodes = nodes.permute(0, 2, 1, 3)  # [B, emb_dim, N, T]
            nodes = self.dropout_node(nodes)
            if self.use_node_gate:
                nodes = self.node_gate(nodes)
            components.append(nodes)

        # 4. Объединение всех компонентов
        x_combined = torch.cat(components, dim=1)

        # 5. Применение общего механизма управления (если есть)
        if self.use_combined_gate:
            x_combined = self.combined_gate(x_combined)

        # 6. Финальное преобразование
        x_out = self.start_conv(x_combined)
        return x_out
    
    def get_filter_reg_loss(self, l2_coeff=1e-4, smooth_coeff=1e-4):
        """Возвращает потери регуляризации фильтра, если фильтр используется"""
        if self.use_filter:
            return self.filter.reg_loss(l2_coeff, smooth_coeff)
        return 0.0

#### GWNet

In [18]:
class EnhancedAdaptiveMatrix(nn.Module):
    def __init__(self, num_nodes, virtual_nodes=5, embedding_dim=32, init_scale=0.1):
        """
        Адаптивная матрица смежности с виртуальными узлами для улучшенной передачи информации.
        
        Args:
            num_nodes: Количество реальных узлов в графе
            virtual_nodes: Количество дополнительных виртуальных узлов
            embedding_dim: Размерность промежуточных эмбеддингов
            init_scale: Масштаб инициализации весов
        """
        super().__init__()
        
        self.num_nodes = num_nodes
        self.virtual_nodes = virtual_nodes
        self.total_nodes = num_nodes + virtual_nodes
        
        # Эмбеддинги для всех узлов (реальных + виртуальных)
        self.node_embeddings1 = nn.Parameter(
            torch.randn(self.total_nodes, embedding_dim)
        )
        self.node_embeddings2 = nn.Parameter(
            torch.randn(embedding_dim, self.total_nodes)
        )
        
        # Параметры для связи виртуальных узлов с реальными
        self.virtual_importance = nn.Parameter(torch.ones(virtual_nodes))
    
    def forward(self):
        """
        Создает адаптивную матрицу смежности.
        
        Returns:
            адаптивная матрица смежности размера (num_nodes, num_nodes)
        """
        # Создаем полную матрицу для всех узлов (реальных + виртуальных)
        full_adj = F.softmax(F.relu(torch.mm(self.node_embeddings1, self.node_embeddings2)), dim=1)
        
        # Извлекаем подматрицы
        real_to_real = full_adj[:self.num_nodes, :self.num_nodes]  # Связи между реальными узлами
        real_to_virtual = full_adj[:self.num_nodes, self.num_nodes:]  # Связи от реальных к виртуальным
        virtual_to_real = full_adj[self.num_nodes:, :self.num_nodes]  # Связи от виртуальных к реальным
        
        # Вычисляем вклад через виртуальные узлы
        # Формула: real_to_real + real_to_virtual @ diag(importance) @ virtual_to_real
        importance_matrix = torch.diag(F.sigmoid(self.virtual_importance))
        virtual_contribution = torch.mm(
            torch.mm(real_to_virtual, importance_matrix),
            virtual_to_real
        )
        
        # Финальная матрица с прямыми связями и связями через виртуальные узлы
        enhanced_adj = real_to_real + virtual_contribution
        
        # Нормализуем
        return F.softmax(enhanced_adj, dim=1)

class ImprovedAdjacencyGenerator(nn.Module):
    def __init__(
        self,
        num_nodes,
        base_adj=None,  # Исходная матрица смежности (при наличии)
        hidden_dim=16,  # Размерность скрытого представления
        temporal_aggregation="attn",  # 'mean', 'max', 'attn'
        combine_mode="additive",  # 'additive', 'convex', 'gated'
        use_spatial_prior=True,  # Использовать ли пространственную близость
        use_temporal_features=True,  # Использовать ли временные признаки
    ):
        super().__init__()
        self.num_nodes = num_nodes
        self.temporal_aggregation = temporal_aggregation
        self.combine_mode = combine_mode
        self.use_spatial_prior = use_spatial_prior
        self.use_temporal_features = use_temporal_features
        
        # 1. Обработка признаков узлов
        self.node_encoder = nn.Sequential(
            nn.Conv2d(1, hidden_dim, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU()
        )
        
        # 2. Временная атенция (если используется)
        if temporal_aggregation == "attn":
            self.time_attention = nn.Sequential(
                nn.Conv2d(hidden_dim, 1, kernel_size=(1, 1)),
                nn.Softmax(dim=-1)
            )
        
        # 3. Генерация смежности из признаков
        self.adj_generator = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # 4. Различные режимы комбинирования с базовой смежностью
        if combine_mode == "convex":
            self.alpha = nn.Parameter(torch.tensor(0.5))
        elif combine_mode == "gated":
            self.gate_net = nn.Sequential(
                nn.Linear(hidden_dim * 2, 1),
                nn.Sigmoid()
            )
        
        # 5. Инициализация базовой смежности 
        if base_adj is not None:
            self.register_buffer('base_adj', base_adj)
        else:
            # Создаем базовую смежность на основе близости индексов
            base = torch.zeros(num_nodes, num_nodes)
            for i in range(num_nodes):
                for j in range(num_nodes):
                    base[i, j] = 1.0 / (1.0 + abs(i - j))
            self.register_buffer('base_adj', base)
            
        # 6. Опциональный временной эмбеддинг
        if use_temporal_features:
            self.time_embed = nn.Sequential(
                nn.Linear(1, hidden_dim),
                nn.ReLU()
            )
    
    def forward(self, X):
        """
        Args:
            X: входной тензор формы (B, C, N, L)
        Returns:
            Динамическую матрицу смежности формы (B, N, N)
        """
        batch_size, channels, nodes, seq_len = X.shape
        
        # 1. Извлекаем основной канал (скорость) и временной признак (если используется)
        speed = X[:, 0:1, :, :]  # Выделяем только канал скорости (первый)
        
        # 2. Кодируем узлы через сверточную сеть
        node_features = self.node_encoder(speed)  # (B, hidden, N, L)
        
        # 3. Агрегация по времени
        if self.temporal_aggregation == "mean":
            node_features = node_features.mean(dim=-1)  # (B, hidden, N)
        elif self.temporal_aggregation == "max": 
            node_features = node_features.max(dim=-1)[0]  # (B, hidden, N)
        else:  # "attn"
            # Вычисляем веса внимания
            attn_weights = self.time_attention(node_features)  # (B, 1, N, L)
            # Взвешенная сумма по времени
            node_features = torch.sum(node_features * attn_weights, dim=-1)  # (B, hidden, N)
        
        # Изменяем форму: (B, hidden, N) -> (B, N, hidden)
        node_features = node_features.transpose(1, 2)
        
        # 4. Добавляем временные признаки (если используются)
        if self.use_temporal_features and X.shape[1] > 1:
            # Предполагаем, что time_feature находится в канале 1
            time_feature = X[:, 1, :, -1].unsqueeze(-1)  # Берем последний шаг (B, N, 1)
            time_emb = self.time_embed(time_feature)  # (B, N, hidden)
            node_features = node_features + time_emb
        
        # 5. Вычисляем матрицу смежности
        # Подготавливаем признаки через MLP
        node_features_transformed = self.adj_generator(node_features)  # (B, N, hidden)
        
        # Вычисляем попарное сходство
        similarity = torch.bmm(
            node_features_transformed, 
            node_features_transformed.transpose(1, 2)
        )  # (B, N, N)
        
        # Применяем активацию и нормализацию
        adj_dynamic = F.softmax(F.relu(similarity), dim=-1)
        
        # 6. Комбинируем с базовой матрицей (если используется)
        if self.use_spatial_prior:
            base_adj_expanded = self.base_adj.expand(batch_size, -1, -1)
            
            if self.combine_mode == "additive":
                # Аддитивная комбинация
                adj_combined = adj_dynamic + 0.1 * base_adj_expanded  # Небольшой вес для базовой
                adj_combined = F.softmax(adj_combined, dim=-1)
            
            elif self.combine_mode == "convex":
                # Выпуклая комбинация с обучаемым параметром
                alpha_clamped = torch.clamp(self.alpha, 0.0, 1.0)
                adj_combined = alpha_clamped * adj_dynamic + (1 - alpha_clamped) * base_adj_expanded
            
            elif self.combine_mode == "gated":
                # Зависимые от контекста веса
                node_features_flat = node_features.reshape(batch_size * nodes, -1)
                base_features = torch.zeros_like(node_features_flat)  # Заглушка для демонстрации
                
                combined_features = torch.cat([node_features_flat, base_features], dim=-1)
                gate = self.gate_net(combined_features).view(batch_size, nodes, 1)
                
                adj_combined = gate * adj_dynamic + (1 - gate) * base_adj_expanded
            
            return adj_combined
        else:
            return adj_dynamic


class LagsExtractor(nn.Module):
    # 3. LagsExtractor: выделение дополнительных каналов (лагов) с дополнением длины последовательности
    def __init__(self, receptive_field, start_channel=3):
        """
        Args:
            receptive_field: минимальная длина последовательности (требуемый receptive field).
            start_channel: индекс, с которого начинаются лаги.
        """
        super().__init__()
        self.receptive_field = receptive_field
        self.start_channel = start_channel

    def forward(self, input_tensor: torch.Tensor):
        """
        Args:
            input_tensor: входной тензор формы (B, C, N, L)
        Returns:
            Лаги или None, если их нет.
        """
        if input_tensor.shape[1] > self.start_channel:
            lags = input_tensor[:, self.start_channel:, :, :]
            seq_len = lags.size(3)
            if seq_len < self.receptive_field:
                lags = F.pad(lags, (self.receptive_field - seq_len, 0, 0, 0))
            return lags
        return None


class AdjacencyMatrixGenerator(nn.Module):
    def __init__(self, num_nodes):
        """
        Args:
            num_nodes: число узлов графа.
        """
        super().__init__()
        self.num_nodes = num_nodes
        self.W = nn.Parameter(torch.randn(num_nodes, num_nodes))
        self.nodevec1 = nn.Parameter(torch.randn(num_nodes, 10))
        self.nodevec2 = nn.Parameter(torch.randn(10, num_nodes))

    def forward(self, X: torch.Tensor):
        """
        Args:
            X: входной тензор формы (B, C, N, L)
        Returns:
            Динамическую матрицу смежности формы (B, N, N)
        """
        # Перестановка: (B, C, N, L) -> (B, N, L, C)
        X_perm = X.permute(0, 2, 3, 1)
        # Используем часть каналов (начиная с 3-го)
        data_window = X_perm[:, :, [0, range(3, X_perm.size(2)+3)], :]
        data_centered = data_window - data_window.mean(dim=2, keepdim=True)
        
        # Compute FFT
        fft_features = torch.fft.rfft(data_centered, dim=2)
        
        # Zero out high-frequency components
        cutoff_freq = int(0.1 * fft_features.size(2))
        fft_features[:, :, cutoff_freq:, :] = 0
        
        # Inverse FFT to get back to the time domain
        filtered_data = torch.fft.irfft(fft_features, dim=2)
        
        # Compute correlation matrix on filtered data
        cov = torch.matmul(filtered_data, filtered_data.transpose(1, 2))
        std = torch.sqrt(torch.sum(filtered_data ** 2, dim=2))
        std_safe = torch.where(std == 0, torch.full_like(std, 1e-8), std)
        corr = cov / (std_safe.unsqueeze(1) * std_safe.unsqueeze(2) + 1e-8)

        # Additional data processing
        additional_data = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
        
        # Combine FFT features and additional data
        combined_data = torch.cat((corr, additional_data.unsqueeze(0).repeat(corr.size(0), 1, 1)), dim=2)
        
        # Compute the final adjacency matrix
        A_dynamic = torch.matmul(torch.matmul(combined_data, self.W), combined_data.transpose(1, 2))
        A_dynamic = F.softmax(F.relu(A_dynamic), dim=-1)
        return A_dynamic
    
class DFDGCNAdjacencyMatrixGenerator(nn.Module):
    def __init__(self, num_nodes, seq_len, embedding_dim=10, time_embedding_dim=12):
        """
        Args:
            num_nodes: Number of nodes in the graph.
            seq_len: Length of the historical time window.
            embedding_dim: Dimension of node embeddings.
            time_embedding_dim: Dimension of time embeddings.
            subgraph_size: Size of the subgraphs for masking (optional).
        """
        super().__init__()
        self.num_nodes = num_nodes
        self.seq_len = seq_len
        self.embedding_dim = embedding_dim
        self.time_embedding_dim = time_embedding_dim

        # Learnable parameters (Identity and Time Embeddings)
        self.node_embeddings = nn.Embedding(num_nodes, embedding_dim)
        self.T_i_D_emb = nn.Embedding(288, time_embedding_dim)  # For Day of Week (assuming 288 steps/day)

        # Projection matrices
        self.Ex1 = nn.Parameter(torch.randn(seq_len // 2 + 1, embedding_dim))  # Adjusted size for frequency domain

        # Convolution and linear layers
        self.Wd = nn.Parameter(torch.randn(embedding_dim + embedding_dim + time_embedding_dim, embedding_dim)) # Adjusted input size
        self.convt = nn.Conv1d(in_channels=embedding_dim, out_channels=embedding_dim, kernel_size=1)
        self.Wxabs = nn.Parameter(torch.randn(embedding_dim, 1))  # For convolution

        #Normalization layers
        self.layersnorm = nn.LayerNorm(embedding_dim)

        # Dropout layer
        self.drop = nn.Dropout(0.5)  # Adjust dropout probability as needed

    def forward(self, input, data):
        """
        Args:
            input: Input tensor of shape (B, C, N, L) - (Batch, Channels, Nodes, Length).  C should be 1.
            data:  Auxiliary data tensor containing time information (day of week, hour of day).  Shape (B, C, N, 2).

        Returns:
            Dynamic adjacency matrix of shape (B, N, N)
        """
        # Construction of dynamic frequency domain graph
        xn1 = input[:, 0, :, -self.seq_len:]  # (B, N, L)

        #Time Embedding
        T_D = self.T_i_D_emb[(data[:, 1, :, 0]).long()]  # (B, N, time_embedding_dim) - Day of Week

        # FFT
        xn1 = torch.fft.rfft(xn1, dim=-1)
        xn1 = torch.abs(xn1)  # (B, N, L//2 + 1)

        # Normalization
        xn1 = F.normalize(xn1, p=2.0, dim=1, eps=1e-12)
        xn1 = F.normalize(xn1, p=2.0, dim=2, eps=1e-12) #* self.a  # Removed self.a (not defined)

        # Projection
        xn1 = torch.matmul(xn1, self.Ex1) # (B, N, embedding_dim)

        # Node Embeddings
        node_embeddings = self.node_embeddings(torch.arange(self.num_nodes, device=input.device)) # (N, embedding_dim)
        xn1k = xn1 + node_embeddings.unsqueeze(0) # (B, N, embedding_dim)

        # Concatenation
        x_n1 = torch.cat([xn1k, T_D], dim=2)  # (B, N, embedding_dim + time_embedding_dim)

        # Linear Layer + Activation + Normalization + Dropout
        x1 = torch.bmm(x_n1, self.Wd)  # (B, N, embedding_dim)
        x1 = torch.relu(x1)
        x1k = self.layersnorm(x1)
        x1k = self.drop(x1k)

        # Adaptive Adjacency Matrix Calculation
        adp = self.convt(x1k.permute(0, 2, 1), self.Wxabs.unsqueeze(0)).squeeze(2) # (B, embedding_dim, N) -> (B, embedding_dim) -> (B, N)
        adj = torch.bmm(x1k, x1k.permute(0, 2, 1)) # (B, N, N)
        adp = torch.relu(adj)

        # Masking and Normalization
        adp = F.softmax(adp, dim=2)

        return adp


class TimeAwareAdaptiveMatrix(nn.Module):
    def __init__(self, num_nodes, num_embeddings=288, emb_dim=32):
        super().__init__()
        self.num_nodes = num_nodes
        
        # Собственный экземпляр TimeEmbedding с большей размерностью
        self.time_embedding = TimeEmbedding(num_embeddings=num_embeddings, emb_dim=emb_dim)
        
        # Базовые эмбеддинги узлов
        self.node_embeddings1 = nn.Parameter(torch.randn(num_nodes, emb_dim))
        # self.node_embeddings2 = nn.Parameter(torch.randn(emb_dim, num_nodes))
        
        # MLP для преобразования временных эмбеддингов в модификаторы графа
        self.time_to_graph = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )
        
    def forward(self, x):
        """
        x: тензор формы (B, C, N, L) где x[:,1,:,:] содержит индексы времени
        """
        batch_size = x.shape[0]
        
        # Извлекаем временные индексы (предполагаем, что они в канале 1)
        time_idx = x[:, 1, :, -1].long()  # Берем последний временной шаг: (B, N)
        
        # Получаем эмбеддинги времени
        # time_emb: (B, emb_dim, N, 1) -> (B, emb_dim, N) -> (B, N, emb_dim)
        time_emb = self.time_embedding(time_idx, target_seq_len=1).squeeze(-1).transpose(1, 2)
        
        # Преобразуем временные эмбеддинги в модификаторы графа
        # Берем только первый узел для времени (предполагаем, что время одинаково для всех узлов)
        time_modifier = self.time_to_graph(time_emb[:, 0, :])  # (B, emb_dim)
        
        adj_batch = []
        for b in range(batch_size):
            # Модифицируем эмбеддинги узлов в зависимости от времени
            # Например, аддитивный подход:
            modified_emb = self.node_embeddings1 + time_modifier[b].unsqueeze(0)
            
            # Вычисляем матрицу смежности
            similarity = torch.mm(modified_emb, modified_emb.t())
            adj = F.softmax(F.relu(similarity), dim=1)
            
            adj_batch.append(adj)
        
        return torch.stack(adj_batch)  # (B, N, N)

# Дополнительные компоненты для графовой свёрточной части (оставлены без изменений)
class nconv(nn.Module):
    """Операция графовой свёртки."""
    def __init__(self):
        super().__init__()

    def forward(self, x, A):
        if len(A.shape) == 2:
            x = torch.einsum('ncvl,vw->ncwl', (x, A))
        elif len(A.shape) == 3:
            x = torch.einsum('ncvl,nvw->ncwl', (x, A))
        return x.contiguous()

class linear(nn.Module):
    """Линейный слой (1x1 свёртка)."""
    def __init__(self, c_in, c_out):
        super().__init__()
        self.mlp = nn.Conv2d(c_in, c_out, kernel_size=(1, 1), bias=True)

    def forward(self, x):
        return self.mlp(x)

class gcn(nn.Module):
    """Графовая свёрточная сеть."""
    def __init__(self, c_in, c_out, dropout, support_len=3, order=2):
        super().__init__()
        self.nconv = nconv()
        c_in_total = (order * support_len + 1) * c_in
        self.mlp = linear(c_in_total, c_out)
        self.dropout = dropout
        self.order = order

    def forward(self, x, support):
        out = [x]
        for a in support:
            x1 = self.nconv(x, a.to(x.device))
            out.append(x1)
            for k in range(2, self.order + 1):
                x1 = self.nconv(x1, a.to(x.device))
                out.append(x1)
        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, self.dropout, training=self.training)
        return h

# 5. Основная модель GWNet с возможностью включения/отключения отдельных компонентов
class GWNet(nn.Module):
    def __init__(self,
                 config,
                 num_nodes,
                 dropout=0.3,
                 supports=None,
                 gcn_bool=True,
                 addaptadj=True,
                 aptinit=None,
                 in_dim=2,
                 out_dim=12,
                 residual_channels=32,
                 dilation_channels=32,
                 skip_channels=256,
                 end_channels=512,
                 kernel_size=2,
                 blocks=4,
                 layers=2,
                 emb_dim=4,
                 use_filter=False,
                 use_time_emb=False,
                 use_dynamic_adj=False,
                 use_nodes=False,
                 use_gate=False,
                 filter_init_range=0.002,
                 ):
        """
        Args:
            num_nodes: число узлов.
            dropout: коэффициент dropout.
            supports: список матриц смежности, если есть.
            gcn_bool: флаг использования GCN.
            addaptadj: флаг использования адаптивной смежности.
            aptinit: инициализация для адаптивной смежности.
            in_dim: число входных каналов.
            out_dim: длина выходной последовательности.
            residual_channels, dilation_channels, skip_channels, end_channels: размеры каналов.
            kernel_size: размер ядра для TCN.
            blocks, layers: структура TCN.
            emb_dim: размерность эмбеддингов времени/узлов.
            add_c: дополнительные каналы.
            use_filter: применять ли LearnableFilter.
            use_time_emb: применять ли TimeEmbedding.
            use_lags: использовать ли выделение lag-показателей.
            use_dynamic_adj: использовать ли динамическую матрицу смежности.
            filter_init_range: диапазон инициализации для фильтра.
        """
        super().__init__()
        self.num_nodes = num_nodes
        self.use_filter = use_filter
        self.use_time_emb = use_time_emb
        self.use_nodes =  use_nodes
        self.use_dynamic_adj = use_dynamic_adj
        self.adapter_required = use_filter or use_time_emb or use_nodes
        self.emb_dim = emb_dim
        self.dropout = dropout
        self.blocks = blocks
        self.layers = layers
        self.gcn_bool = gcn_bool
        self.addaptadj = addaptadj
        self.window_size = 12  # можно сделать параметром, если потребуется
        extra_channels = 0

        self.start_conv = nn.Conv2d(in_channels=in_dim + extra_channels,
                                    out_channels=residual_channels,
                                    kernel_size=(1, 1))

        if self.adapter_required:
            self.adapter = UniversalInputAdapter(
                seq_len=self.window_size,
                n_nodes=num_nodes,
                residual_channels=residual_channels,
                use_filter=use_filter,
                use_time_emb=use_time_emb,
                use_nodes=use_nodes,
                use_filter_gate=config.get('use_filter_gate', False),
                use_time_gate=config.get('use_time_gate', False),
                use_node_gate=config.get('use_node_gate', False),
                use_combined_gate=config.get('use_combined_gate', False),
                filter_gate_type=config.get('filter_gate_type', 'frequency'),
                time_gate_type=config.get('time_gate_type', 'spatiotemporal'),
                node_gate_type=config.get('node_gate_type', 'node'),
                combined_gate_type=config.get('combined_gate_type', 'multiresolution'),
                dropout_rate=config.get('dropout_rate', 0.1),
                freq_mask_threshold=config.get('freq_mask_threshold', None),
                time_emb_dim=config.get('time_emb_dim', 10),
                node_emb_dim=config.get('node_emb_dim', 10)
            )
        else:
            self.adapter = None


        self.supports = supports
        self.supports_len = 0
        if supports is not None:
            self.supports_len += len(supports)
        if gcn_bool and addaptadj:
            if aptinit is None:
                self.nodevec1 = nn.Parameter(torch.randn(num_nodes, 10))
                self.nodevec2 = nn.Parameter(torch.randn(10, num_nodes))
                self.supports_len += 1
                if self.use_dynamic_adj:
                    self.supports_len += 0
            else:
                if supports is None:
                    supports = []
                m, p, n = torch.svd(aptinit)
                initemb1 = torch.mm(m[:, :10], torch.diag(p[:10] ** 0.5))
                initemb2 = torch.mm(torch.diag(p[:10] ** 0.5), n[:, :10].t())
                self.nodevec1 = nn.Parameter(initemb1)
                self.nodevec2 = nn.Parameter(initemb2)
                self.supports_len += 1
                if self.use_dynamic_adj:
                    self.supports_len += 0

        # Инициализация модулей TCN и GCN
        self.filter_convs = nn.ModuleList()
        self.gate_convs = nn.ModuleList()
        self.residual_convs = nn.ModuleList()
        self.skip_convs = nn.ModuleList()
        self.bn = nn.ModuleList()
        self.gconv = nn.ModuleList()

        receptive_field = 1

        for b in range(blocks):
            additional_scope = kernel_size - 1
            new_dilation = 1
            for i in range(layers):
                self.filter_convs.append(nn.Conv2d(in_channels=residual_channels,
                                                   out_channels=dilation_channels,
                                                   kernel_size=(1, kernel_size),
                                                   dilation=new_dilation))
                self.gate_convs.append(nn.Conv2d(in_channels=residual_channels,
                                                 out_channels=dilation_channels,
                                                 kernel_size=(1, kernel_size),
                                                 dilation=new_dilation))
                self.residual_convs.append(nn.Conv2d(in_channels=dilation_channels,
                                                     out_channels=residual_channels,
                                                     kernel_size=(1, 1)))
                self.skip_convs.append(nn.Conv2d(in_channels=dilation_channels,
                                                 out_channels=skip_channels,
                                                 kernel_size=(1, 1)))
                self.bn.append(nn.BatchNorm2d(residual_channels))
                new_dilation *= 2
                receptive_field += additional_scope
                additional_scope *= 2
                if self.gcn_bool:
                    self.gconv.append(gcn(dilation_channels, residual_channels, dropout, support_len=self.supports_len))
        self.receptive_field = receptive_field

        self.end_conv_1 = nn.Conv2d(in_channels=skip_channels,
                                    out_channels=end_channels,
                                    kernel_size=(1, 1))
        self.end_conv_2 = nn.Conv2d(in_channels=end_channels,
                                    out_channels=out_dim,
                                    kernel_size=(1, 1))


    def compute_adaptive_supports(self, input_tensor):
        new_supports = None
        if self.gcn_bool and self.addaptadj and self.supports is not None:
            adp1 = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
            new_supports = self.supports + [adp1]
        return new_supports

    def forward(self, history_data: torch.Tensor) -> torch.Tensor:
        """
        Args:
            history_data: входной тензор формы (B, L, N, C)
        Returns:
            Выходной тензор формы (B, out_dim, N, 1)
        """
        x = history_data.transpose(1, 3).contiguous()

        if self.adapter is not None:
            x_combined = self.adapter(history_data)
            # shape: torch.Size([16, 3, 307, 12]) [B, C, N, L]

            # x_combined = x_combined.permute(0, 1, 3, 2).contiguous()
            in_len = x.size(3)
            if in_len < self.receptive_field:
                x_combined = F.pad(x_combined, (self.receptive_field - in_len, 0, 0, 0))
                            
        else:
            in_len = x.size(3)
            if in_len < self.receptive_field:
                x_combined = F.pad(x, (self.receptive_field - in_len, 0, 0, 0))

            x_combined = self.start_conv(x_combined)

        skip = 0
        new_supports = self.compute_adaptive_supports(x)

        # TCN + GCN блоки
        for i in range(self.blocks * self.layers):
            residual = x_combined
            filter_out = torch.tanh(self.filter_convs[i](residual))
            gate_out = torch.sigmoid(self.gate_convs[i](residual))
            x_combined = filter_out * gate_out

            s = self.skip_convs[i](x_combined)
            if isinstance(skip, torch.Tensor):
                # Усечение skip, если требуется
                skip = skip[:, :, :, -s.size(3):]
            else:
                skip = 0
            skip = s + skip

            # Применяем либо GCN, либо обычную свёрточную сеть для остаточного соединения
            if self.gcn_bool and self.supports is not None:
                if self.addaptadj:
                    x_combined = self.gconv[i](x_combined, new_supports)
                else:
                    x_combined = self.gconv[i](x_combined, self.supports)
            else:
                x_combined = self.residual_convs[i](x_combined)

            # Остаточное соединение с усечением по длине
            x_combined = x_combined + residual[:, :, :, -x_combined.size(3):]
            x_combined = self.bn[i](x_combined)

        out = F.relu(skip)
        out = F.relu(self.end_conv_1(out))
        out = self.end_conv_2(out)
        return out


#### STGCN

In [19]:
class Align(nn.Module):
    def __init__(self, c_in, c_out):
        super(Align, self).__init__()
        self.c_in = c_in
        self.c_out = c_out
        self.align_conv = nn.Conv2d(
            in_channels=c_in, out_channels=c_out, kernel_size=(1, 1))

    def forward(self, x):
        if self.c_in > self.c_out:
            x = self.align_conv(x)
        elif self.c_in < self.c_out:
            batch_size, _, timestep, n_vertex = x.shape
            x = torch.cat([x, torch.zeros(
                [batch_size, self.c_out - self.c_in, timestep, n_vertex]).to(x)], dim=1)
        else:
            x = x

        return x


class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, enable_padding=False, dilation=1, groups=1, bias=True):
        if enable_padding == True:
            self.__padding = (kernel_size - 1) * dilation
        else:
            self.__padding = 0
        super(CausalConv1d, self).__init__(in_channels, out_channels, kernel_size=kernel_size,
                                           stride=stride, padding=self.__padding, dilation=dilation, groups=groups, bias=bias)

    def forward(self, input):
        result = super(CausalConv1d, self).forward(input)
        if self.__padding != 0:
            return result[:, :, : -self.__padding]

        return result


class CausalConv2d(nn.Conv2d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, enable_padding=False, dilation=1, groups=1, bias=True):
        kernel_size = nn.modules.utils._pair(kernel_size)
        stride = nn.modules.utils._pair(stride)
        dilation = nn.modules.utils._pair(dilation)
        if enable_padding == True:
            self.__padding = [int((kernel_size[i] - 1) * dilation[i])
                              for i in range(len(kernel_size))]
        else:
            self.__padding = 0
        self.left_padding = nn.modules.utils._pair(self.__padding)
        super(CausalConv2d, self).__init__(in_channels, out_channels, kernel_size,
                                           stride=stride, padding=0, dilation=dilation, groups=groups, bias=bias)

    def forward(self, input):
        if self.__padding != 0:
            input = F.pad(
                input, (self.left_padding[1], 0, self.left_padding[0], 0))
        result = super(CausalConv2d, self).forward(input)

        return result


class TemporalConvLayer(nn.Module):

    # Temporal Convolution Layer (GLU)
    #
    #        |--------------------------------| * Residual Connection *
    #        |                                |
    #        |    |--->--- CasualConv2d ----- + -------|
    # -------|----|                                   ⊙ ------>
    #             |--->--- CasualConv2d --- Sigmoid ---|
    #

    # param x: tensor, [bs, c_in, ts, n_vertex]

    def __init__(self, Kt, c_in, c_out, n_vertex, act_func):
        super(TemporalConvLayer, self).__init__()
        self.Kt = Kt
        self.c_in = c_in
        self.c_out = c_out
        self.n_vertex = n_vertex
        self.align = Align(c_in, c_out)
        if act_func == 'glu' or act_func == 'gtu':
            self.causal_conv = CausalConv2d(
                in_channels=c_in, out_channels=2 * c_out, kernel_size=(Kt, 1), enable_padding=False, dilation=1)
        else:
            self.causal_conv = CausalConv2d(in_channels=c_in, out_channels=c_out, kernel_size=(
                Kt, 1), enable_padding=False, dilation=1)
        self.act_func = act_func
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()
        self.relu = nn.ReLU()
        self.leaky_relu = nn.LeakyReLU()
        self.silu = nn.SiLU()

    def forward(self, x):
        
        x_in = self.align(x)[:, :, self.Kt - 1:, :]
        x_causal_conv = self.causal_conv(x)

        if self.act_func == 'glu' or self.act_func == 'gtu':
            x_p = x_causal_conv[:, : self.c_out, :, :]
            x_q = x_causal_conv[:, -self.c_out:, :, :]

            if self.act_func == 'glu':
                # GLU was first purposed in
                # *Language Modeling with Gated Convolutional Networks*.
                # URL: https://arxiv.org/abs/1612.08083
                # Input tensor X is split by a certain dimension into tensor X_a and X_b.
                # In the original paper, GLU is defined as Linear(X_a) ⊙ Sigmoid(Linear(X_b)).
                # However, in PyTorch, GLU is defined as X_a ⊙ Sigmoid(X_b).
                # URL: https://pytorch.org/docs/master/nn.functional.html#torch.nn.functional.glu
                # Because in original paper, the representation of GLU and GTU is ambiguous.
                # So, it is arguable which one version is correct.

                # (x_p + x_in) ⊙ Sigmoid(x_q)
                x = torch.mul((x_p + x_in), self.sigmoid(x_q))

            else:
                # Tanh(x_p + x_in) ⊙ Sigmoid(x_q)
                x = torch.mul(self.tanh(x_p + x_in), self.sigmoid(x_q))

        elif self.act_func == 'relu':
            x = self.relu(x_causal_conv + x_in)

        elif self.act_func == 'leaky_relu':
            x = self.leaky_relu(x_causal_conv + x_in)

        elif self.act_func == 'silu':
            x = self.silu(x_causal_conv + x_in)

        else:
            raise NotImplementedError(
                f'ERROR: The activation function {self.act_func} is not implemented.')

        return x


class ChebGraphConv(nn.Module):
    def __init__(self, c_in, c_out, Ks, gso, bias):
        super(ChebGraphConv, self).__init__()
        self.c_in = c_in
        self.c_out = c_out
        self.Ks = Ks
        self.gso = gso
        self.weight = nn.Parameter(torch.FloatTensor(Ks, c_in, c_out))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(c_out))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        #bs, c_in, ts, n_vertex = x.shape
        x = torch.permute(x, (0, 2, 3, 1))

        self.gso = self.gso.to(x.device)

        if self.Ks - 1 < 0:
            raise ValueError(
                f'ERROR: the graph convolution kernel size Ks has to be a positive integer, but received {self.Ks}.')
        elif self.Ks - 1 == 0:
            x_0 = x
            x_list = [x_0]
        elif self.Ks - 1 == 1:
            x_0 = x
            x_1 = torch.einsum('hi,btij->bthj', self.gso, x)
            x_list = [x_0, x_1]
        elif self.Ks - 1 >= 2:
            x_0 = x
            x_1 = torch.einsum('hi,btij->bthj', self.gso, x)
            x_list = [x_0, x_1]
            for k in range(2, self.Ks):
                x_list.append(torch.einsum('hi,btij->bthj', 2 *
                              self.gso, x_list[k - 1]) - x_list[k - 2])

        x = torch.stack(x_list, dim=2)

        cheb_graph_conv = torch.einsum('btkhi,kij->bthj', x, self.weight)

        if self.bias is not None:
            cheb_graph_conv = torch.add(cheb_graph_conv, self.bias)
        else:
            cheb_graph_conv = cheb_graph_conv

        return cheb_graph_conv


class GraphConv(nn.Module):
    def __init__(self, c_in, c_out, gso, bias):
        super(GraphConv, self).__init__()
        self.c_in = c_in
        self.c_out = c_out
        self.gso = gso
        self.weight = nn.Parameter(torch.FloatTensor(c_in, c_out))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(c_out))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        #bs, c_in, ts, n_vertex = x.shape
        x = torch.permute(x, (0, 2, 3, 1))

        self.gso = self.gso.to(x.device)

        first_mul = torch.einsum('hi,btij->bthj', self.gso, x)
        second_mul = torch.einsum('bthi,ij->bthj', first_mul, self.weight)

        if self.bias is not None:
            graph_conv = torch.add(second_mul, self.bias)
        else:
            graph_conv = second_mul

        return graph_conv


class GraphConvLayer(nn.Module):
    def __init__(self, graph_conv_type, c_in, c_out, Ks, gso, bias):
        super(GraphConvLayer, self).__init__()
        self.graph_conv_type = graph_conv_type
        self.c_in = c_in
        self.c_out = c_out
        self.align = Align(c_in, c_out)
        self.Ks = Ks
        self.gso = gso
        if self.graph_conv_type == 'cheb_graph_conv':
            self.cheb_graph_conv = ChebGraphConv(c_out, c_out, Ks, gso, bias)
        elif self.graph_conv_type == 'graph_conv':
            self.graph_conv = GraphConv(c_out, c_out, gso, bias)

    def forward(self, x):
        x_gc_in = self.align(x)
        if self.graph_conv_type == 'cheb_graph_conv':
            x_gc = self.cheb_graph_conv(x_gc_in)
        elif self.graph_conv_type == 'graph_conv':
            x_gc = self.graph_conv(x_gc_in)
        x_gc = x_gc.permute(0, 3, 1, 2)
        x_gc_out = torch.add(x_gc, x_gc_in)

        return x_gc_out


class STConvBlock(nn.Module):
    # STConv Block contains 'TGTND' structure
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # G: Graph Convolution Layer (ChebGraphConv or GraphConv)
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normolization
    # D: Dropout

    def __init__(self, Kt, Ks, n_vertex, last_block_channel, channels, act_func, graph_conv_type, gso, bias, droprate):
        super(STConvBlock, self).__init__()

        self.tmp_conv1 = TemporalConvLayer(
            Kt, last_block_channel, channels[0], n_vertex, act_func)
        self.graph_conv = GraphConvLayer(
            graph_conv_type, channels[0], channels[1], Ks, gso, bias)
        self.tmp_conv2 = TemporalConvLayer(
            Kt, channels[1], channels[2], n_vertex, act_func)
        self.tc2_ln = nn.LayerNorm([n_vertex, channels[2]])
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=droprate)

    def forward(self, x):
        
        x = self.tmp_conv1(x)
        x = self.graph_conv(x)
        x = self.relu(x)
        x = self.tmp_conv2(x)
        x = self.tc2_ln(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        x = self.dropout(x)

        return x


class OutputBlock(nn.Module):
    # Output block contains 'TNFF' structure
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normolization
    # F: Fully-Connected Layer
    # F: Fully-Connected Layer

    def __init__(self, Ko, last_block_channel, channels, end_channel, n_vertex, act_func, bias, droprate):
        super(OutputBlock, self).__init__()
        self.tmp_conv1 = TemporalConvLayer(
            Ko, last_block_channel, channels[0], n_vertex, act_func)
        self.fc1 = nn.Linear(
            in_features=channels[0], out_features=channels[1], bias=bias)
        self.fc2 = nn.Linear(
            in_features=channels[1], out_features=end_channel, bias=bias)
        self.tc1_ln = nn.LayerNorm([n_vertex, channels[0]])
        self.relu = nn.ReLU()
        self.leaky_relu = nn.LeakyReLU()
        self.silu = nn.SiLU()
        self.dropout = nn.Dropout(p=droprate)

    def forward(self, x):
        x = self.tmp_conv1(x)
        x = self.tc1_ln(x.permute(0, 2, 3, 1))
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x).permute(0, 3, 1, 2)

        return x


class STGCNChebGraphConv(nn.Module):
    """
    Paper: Spatio-Temporal Graph Convolutional Networks: A Deep Learning Framework for Trafﬁc Forecasting
    Official Code: https://github.com/VeritasYin/STGCN_IJCAI-18 (tensorflow)
    Ref Code: https://github.com/hazdzz/STGCN
    Venue: IJCAI 2018
    Task: Spatial-Temporal Forecasting
    Note:  
        https://github.com/hazdzz/STGCN/issues/9
    Link: https://arxiv.org/abs/1709.04875
    """

    # STGCNChebGraphConv contains 'TGTND TGTND TNFF' structure
    # ChebGraphConv is the graph convolution from ChebyNet.
    # Using the Chebyshev polynomials of the first kind as a graph filter.

    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # G: Graph Convolution Layer (ChebGraphConv)
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normolization
    # D: Dropout

    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # G: Graph Convolution Layer (ChebGraphConv)
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normolization
    # D: Dropout

    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normalization
    # F: Fully-Connected Layer
    # F: Fully-Connected Layer

    def __init__(self,
                 config,
                 n_vertex, 
                 gso, 
                 Kt=3, 
                 Ks=3, 
                 blocks=[[1], [32, 16, 32], [32, 16, 32], [64, 64], [12]], 
                 T=12, 
                 act_func = "glu", 
                 graph_conv_type = "cheb_graph_conv", 
                 bias = True, 
                 droprate = 0.5,
                 residual_channels=32, 
                 use_filter=False, 
                 use_time_emb=False, 
                 use_nodes=False,
                 dropout_rate=0.5):
        super(STGCNChebGraphConv, self).__init__()

        self.use_filter = use_filter
        self.use_time_emb = use_time_emb
        self.use_nodes =  use_nodes
        self.adapter_required = use_filter or use_time_emb or use_nodes

        modules = []
        for l in range(len(blocks) - 3):
            in_channels = residual_channels if self.adapter_required and l == 0 else blocks[l][-1]
            modules.append(STConvBlock(
                Kt, Ks, n_vertex, in_channels, blocks[l + 1],
                act_func, graph_conv_type, gso, bias, droprate
            ))
        self.st_blocks = nn.Sequential(*modules)
        Ko = T - (len(blocks) - 3) * 2 * (Kt - 1)
        self.Ko = Ko
        assert Ko != 0, "Ko = 0."
        self.output = OutputBlock(
            Ko, blocks[-3][-1], blocks[-2], blocks[-1][0], n_vertex, act_func, bias, droprate)
        
        if self.adapter_required:
            self.adapter = UniversalInputAdapter(
                seq_len=T,
                n_nodes=n_vertex,
                residual_channels=residual_channels,
                use_filter=use_filter,
                use_time_emb=use_time_emb,
                use_nodes=use_nodes,
                use_filter_gate=config.get('use_filter_gate', False),
                use_time_gate=config.get('use_time_gate', False),
                use_node_gate=config.get('use_node_gate', False),
                use_combined_gate=config.get('use_combined_gate', False),
                filter_gate_type=config.get('filter_gate_type', 'frequency'),
                time_gate_type=config.get('time_gate_type', 'spatiotemporal'),
                node_gate_type=config.get('node_gate_type', 'node'),
                combined_gate_type=config.get('combined_gate_type', 'multiresolution'),
                dropout_rate=config.get('dropout_rate', 0.1),
                freq_mask_threshold=config.get('freq_mask_threshold', None),
                time_emb_dim=config.get('time_emb_dim', 10),
                node_emb_dim=config.get('node_emb_dim', 10)
            )
        else:
            self.adapter = None

    def forward(self, history_data: torch.Tensor) -> torch.Tensor:
        """feedforward function of STGCN.

        Args:
            history_data (torch.Tensor): historical data with shape [B, L, N, C]

        Returns:
            torch.Tensor: prediction with shape [B, L, N, C]
        """

# x_combined: torch.Size([13, 1, 12, 307])
                                                                    

# === Запуск конфигурации: {'use_filter': False, 'use_nodes': False, 'use_time_emb': True} ===
# ===(1/8) Запуск конфигурации: {'use_filter_gate': True, 'use_time_gate': True, 'use_node_gate': True} ===
# x_combined: torch.Size([16, 32, 307, 12])


        if self.adapter_required:
            x = history_data#.permute(0, 3, 1, 2).contiguous()
            x_combined = self.adapter(x).permute(0, 1, 3, 2).contiguous()
        else:
            x_combined = history_data[:, :, :, [0]].permute(0, 3, 1, 2).contiguous()

        x_combined = self.st_blocks(x_combined)
        x_combined = self.output(x_combined)

        x_combined = x_combined.transpose(2, 3)
        return x_combined
    


#### AGCRN

In [20]:
class AVWGCN(nn.Module):
    def __init__(self, dim_in, dim_out, cheb_k, embed_dim):
        super(AVWGCN, self).__init__()
        self.cheb_k = cheb_k
        self.weights_pool = nn.Parameter(
            torch.FloatTensor(embed_dim, cheb_k, dim_in, dim_out))
        self.bias_pool = nn.Parameter(torch.FloatTensor(embed_dim, dim_out))

    def forward(self, x, node_embeddings):
        # x shaped[B, N, C], node_embeddings shaped [N, D] -> supports shaped [N, N]
        # output shape [B, N, C]
        node_num = node_embeddings.shape[0]
        supports = F.softmax(
            F.relu(torch.mm(node_embeddings, node_embeddings.transpose(0, 1))), dim=1)
        support_set = [torch.eye(node_num).to(supports.device), supports]
        # default cheb_k = 3
        for k in range(2, self.cheb_k):
            support_set.append(torch.matmul(
                2 * supports, support_set[-1]) - support_set[-2])
        supports = torch.stack(support_set, dim=0)
        # N, cheb_k, dim_in, dim_out
        weights = torch.einsum(
            'nd,dkio->nkio', node_embeddings, self.weights_pool)
        bias = torch.matmul(node_embeddings, self.bias_pool)  # N, dim_out
        x_g = torch.einsum("knm,bmc->bknc", supports,
                           x)  # B, cheb_k, N, dim_in
        x_g = x_g.permute(0, 2, 1, 3)  # B, N, cheb_k, dim_in
        x_gconv = torch.einsum('bnki,nkio->bno', x_g,
                               weights) + bias  # b, N, dim_out
        return x_gconv

class AGCRNCell(nn.Module):
    def __init__(self, node_num, dim_in, dim_out, cheb_k, embed_dim):
        super(AGCRNCell, self).__init__()
        self.node_num = node_num
        self.hidden_dim = dim_out
        self.gate = AVWGCN(dim_in+self.hidden_dim, 2 *
                           dim_out, cheb_k, embed_dim)
        self.update = AVWGCN(dim_in+self.hidden_dim,
                             dim_out, cheb_k, embed_dim)

    def forward(self, x, state, node_embeddings):
        # x: B, num_nodes, input_dim
        # state: B, num_nodes, hidden_dim
        state = state.to(x.device)
        input_and_state = torch.cat((x, state), dim=-1)
        z_r = torch.sigmoid(self.gate(input_and_state, node_embeddings))
        z, r = torch.split(z_r, self.hidden_dim, dim=-1)
        candidate = torch.cat((x, z*state), dim=-1)
        hc = torch.tanh(self.update(candidate, node_embeddings))
        h = r*state + (1-r)*hc
        return h

    def init_hidden_state(self, batch_size):
        return torch.zeros(batch_size, self.node_num, self.hidden_dim)

class AVWDCRNN(nn.Module):
    def __init__(self, node_num, dim_in, dim_out, cheb_k, embed_dim, num_layers=1):
        super(AVWDCRNN, self).__init__()
        assert num_layers >= 1, 'At least one DCRNN layer in the Encoder.'
        self.node_num = node_num
        self.input_dim = dim_in
        self.num_layers = num_layers
        self.dcrnn_cells = nn.ModuleList()
        self.dcrnn_cells.append(
            AGCRNCell(node_num, dim_in, dim_out, cheb_k, embed_dim))
        for _ in range(1, num_layers):
            self.dcrnn_cells.append(
                AGCRNCell(node_num, dim_out, dim_out, cheb_k, embed_dim))

    def forward(self, x, init_state, node_embeddings):
        # shape of x: (B, T, N, D)
        # shape of init_state: (num_layers, B, N, hidden_dim)
        assert x.shape[2] == self.node_num and x.shape[3] == self.input_dim, f'x has wrong shape: {x.shape}'
        seq_length = x.shape[1]
        current_inputs = x
        output_hidden = []
        for i in range(self.num_layers):
            state = init_state[i]
            inner_states = []
            for t in range(seq_length):
                state = self.dcrnn_cells[i](
                    current_inputs[:, t, :, :], state, node_embeddings)
                inner_states.append(state)
            output_hidden.append(state)
            current_inputs = torch.stack(inner_states, dim=1)
        # current_inputs: the outputs of last layer: (B, T, N, hidden_dim)
        # output_hidden: the last state for each layer: (num_layers, B, N, hidden_dim)
        #last_state: (B, N, hidden_dim)
        return current_inputs, output_hidden

    def init_hidden(self, batch_size):
        init_states = []
        for i in range(self.num_layers):
            init_states.append(
                self.dcrnn_cells[i].init_hidden_state(batch_size))
        # (num_layers, B, N, hidden_dim)
        return torch.stack(init_states, dim=0)

class AGCRN(nn.Module):
    """
    Paper: Adaptive Graph Convolutional Recurrent Network for Trafﬁc Forecasting
    Official Code: https://github.com/LeiBAI/AGCRN
    Link: https://arxiv.org/abs/2007.02842
    Venue: NeurIPS 2020
    Task: Spatial-Temporal Forecasting
    """

    def __init__(self, 
                 config,
                 num_nodes, 
                 input_dim, 
                 rnn_units, 
                 output_dim, 
                 seq_len, 
                 horizon, 
                 num_layers, 
                 default_graph, 
                 embed_dim, 
                 cheb_k,
                 use_nodes=False,
                 use_filter=False,
                 use_time_emb=False,
                 residual_channels=32,
                 dropout_rate=0.2,
                 ):
        super(AGCRN, self).__init__()
        self.num_node = num_nodes
        self.input_dim = input_dim
        self.hidden_dim = rnn_units
        self.output_dim = output_dim
        self.horizon = horizon
        self.num_layers = num_layers

        self.use_filter = use_filter
        self.use_time_emb = use_time_emb
        self.use_nodes =  use_nodes
        self.adapter_required = use_filter or use_time_emb or use_nodes
        
        if self.adapter_required:
            self.adapter = UniversalInputAdapter(
                seq_len=seq_len,
                n_nodes=num_nodes,
                residual_channels=residual_channels,
                use_filter=use_filter,
                use_time_emb=use_time_emb,
                use_nodes=use_nodes,
                use_filter_gate=config.get('use_filter_gate', False),
                use_time_gate=config.get('use_time_gate', False),
                use_node_gate=config.get('use_node_gate', False),
                use_combined_gate=config.get('use_combined_gate', False),
                filter_gate_type=config.get('filter_gate_type', 'frequency'),
                time_gate_type=config.get('time_gate_type', 'spatiotemporal'),
                node_gate_type=config.get('node_gate_type', 'node'),
                combined_gate_type=config.get('combined_gate_type', 'multiresolution'),
                dropout_rate=config.get('dropout_rate', 0.1),
                freq_mask_threshold=config.get('freq_mask_threshold', None),
                time_emb_dim=config.get('time_emb_dim', 10),
                node_emb_dim=config.get('node_emb_dim', 10)
            )
        else:
            self.adapter = None

        self.default_graph = default_graph
        self.node_embeddings = nn.Parameter(torch.randn(
            self.num_node, embed_dim), requires_grad=True)


        self.encoder = AVWDCRNN(num_nodes, 32 if self.adapter_required else input_dim, rnn_units, cheb_k,
                                embed_dim, num_layers)

        # predictor
        self.end_conv = nn.Conv2d(
            1, horizon * self.output_dim, kernel_size=(1, self.hidden_dim), bias=True)

        self.init_param()

    def init_param(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
            else:
                nn.init.uniform_(p)

    def forward(self, history_data: torch.Tensor) -> torch.Tensor:
        """Feedforward function of AGCRN.

        Args:
            history_data (torch.Tensor): inputs with shape [B, L, N, C].

        Returns:
            torch.Tensor: outputs with shape [B, L, N, C]
        """
        if self.adapter_required:
            history_data = self.adapter(history_data) # [B, C, N, L]
            history_data = history_data.permute(0,3,2,1).contiguous()
        else:
            history_data = history_data[:, :, :, [0]]

        init_state = self.encoder.init_hidden(history_data.shape[0])
        output, _ = self.encoder(
            history_data, init_state, self.node_embeddings)  # B, T, N, hidden
        output = output[:, -1:, :, :]  # B, 1, N, hidden

        # CNN based predictor
        output = self.end_conv((output))  # B, T*C, N, 1
        output = output.squeeze(-1).reshape(-1, self.horizon,
                                            self.output_dim, self.num_node)
        output = output.permute(0, 1, 3, 2)  # B, T, N, C

        return output
    

#### Metrics

In [21]:
def compute_metrics(output, target):
    """Вычисление MAE, RMSE, MAPE."""
    diff = output - target
    mae = torch.abs(diff).mean().item()
    rmse = torch.sqrt((diff ** 2).mean()).item()

    denominator = torch.abs(target)
    mask = denominator != 0
    if mask.sum() > 0:
        mape = (torch.abs(diff[mask]) / denominator[mask]).mean().item()
    else:
        mape = 0.0

    return mae, rmse, mape


def train_val_test_model(model, optimizer, criterion, train_loader, val_loader, test_loader, epochs, writer):
    best_val_loss = float('inf')

    for epoch in range(epochs):
        # === Тренировка ===
        model.train()
        train_loss, train_mae, train_rmse, train_mape = 0.0, 0.0, 0.0, 0.0
        train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} - Training", leave=False)

        for step, (x, y) in enumerate(train_loader_tqdm):
            optimizer.zero_grad()
            output = model(x).squeeze(-1)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            batch_size = x.size(0)
            train_loss += loss.item() * batch_size

            mae, rmse, mape = compute_metrics(output, y)
            train_mae += mae * batch_size
            train_rmse += rmse * batch_size
            train_mape += mape * batch_size

            step_global = epoch * len(train_loader) + step
            writer.add_scalar("Loss/Train", loss.item(), step_global)
            writer.add_scalar("MAE/Train", mae, step_global)
            writer.add_scalar("RMSE/Train", rmse, step_global)
            writer.add_scalar("MAPE/Train", mape, step_global)

        total_samples = len(train_loader.dataset)
        train_loss /= total_samples
        train_mae /= total_samples
        train_rmse /= total_samples
        train_mape /= total_samples

        writer.add_scalar("Loss/Train_Avg", train_loss, epoch + 1)
        writer.add_scalar("MAE/Train_Avg", train_mae, epoch + 1)
        writer.add_scalar("RMSE/Train_Avg", train_rmse, epoch + 1)
        writer.add_scalar("MAPE/Train_Avg", train_mape, epoch + 1)

        # === Валидация ===
        model.eval()
        val_loss, val_mae, val_rmse, val_mape = 0.0, 0.0, 0.0, 0.0
        val_loader_tqdm = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{epochs} - Validation", leave=False)

        with torch.no_grad():
            for step, (x, y) in enumerate(val_loader_tqdm):
                output = model(x).squeeze(-1)
                loss = criterion(output, y)

                batch_size = x.size(0)
                val_loss += loss.item() * batch_size

                mae, rmse, mape = compute_metrics(output, y)
                val_mae += mae * batch_size
                val_rmse += rmse * batch_size
                val_mape += mape * batch_size

                step_global = epoch * len(val_loader) + step
                writer.add_scalar("Loss/Validation", loss.item(), step_global)
                writer.add_scalar("MAE/Validation", mae, step_global)
                writer.add_scalar("RMSE/Validation", rmse, step_global)
                writer.add_scalar("MAPE/Validation", mape, step_global)

        total_samples = len(val_loader.dataset)
        val_loss /= total_samples
        val_mae /= total_samples
        val_rmse /= total_samples
        val_mape /= total_samples

        writer.add_scalar("Loss/Validation_Avg", val_loss, epoch + 1)
        writer.add_scalar("MAE/Validation_Avg", val_mae, epoch + 1)
        writer.add_scalar("RMSE/Validation_Avg", val_rmse, epoch + 1)
        writer.add_scalar("MAPE/Validation_Avg", val_mape, epoch + 1)

        # === Тестирование ===
        test_loss, test_mae, test_rmse, test_mape = 0.0, 0.0, 0.0, 0.0
        test_loader_tqdm = tqdm(test_loader, desc=f"Epoch {epoch + 1}/{epochs} - Testing", leave=False)

        with torch.no_grad():
            for step, (x, y) in enumerate(test_loader_tqdm):
                output = model(x).squeeze(-1)
                loss = criterion(output, y)

                batch_size = x.size(0)
                test_loss += loss.item() * batch_size

                mae, rmse, mape = compute_metrics(output, y)
                test_mae += mae * batch_size
                test_rmse += rmse * batch_size
                test_mape += mape * batch_size

                step_global = epoch * len(test_loader) + step
                writer.add_scalar("Loss/Test", loss.item(), step_global)
                writer.add_scalar("MAE/Test", mae, step_global)
                writer.add_scalar("RMSE/Test", rmse, step_global)
                writer.add_scalar("MAPE/Test", mape, step_global)

        total_samples = len(test_loader.dataset)
        test_loss /= total_samples
        test_mae /= total_samples
        test_rmse /= total_samples
        test_mape /= total_samples

        writer.add_scalar("Loss/Test_Avg", test_loss, epoch + 1)
        writer.add_scalar("MAE/Test_Avg", test_mae, epoch + 1)
        writer.add_scalar("RMSE/Test_Avg", test_rmse, epoch + 1)
        writer.add_scalar("MAPE/Test_Avg", test_mape, epoch + 1)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f"{writer.log_dir}/best_model.pth")

    writer.close()


#### Train

In [22]:
# === Устройство ===
device = torch.device("cuda") if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# === Гиперпараметры ===
num_nodes = train_data.shape[1]

# === Функция для инициализации модели ===
def create_gwnet_model(config, use_filter=False, use_time_emb=False, use_dynamic_adj=False, use_nodes=False):
    emb_dim = 4 if use_time_emb else 0
    return GWNet(
        config=config,
        num_nodes=num_nodes,
        in_dim=3,
        emb_dim=emb_dim,
        use_nodes=use_nodes,
        use_filter=use_filter,
        use_time_emb=use_time_emb,
        use_dynamic_adj=use_dynamic_adj, 
        residual_channels=32,
        dilation_channels=32,
        skip_channels=256,
        end_channels=512
        ).to(device)

def create_stgcn_model(config, gso, use_filter=False, use_time_emb=False, use_nodes=False):
    return STGCNChebGraphConv(
        config=config, 
        n_vertex=num_nodes,
        gso=gso,
        blocks=[[1], [16, 4, 16], [16, 4, 16], [32, 32], [12]],
        use_filter=use_filter,
        use_time_emb=use_time_emb,
        use_nodes=use_nodes
        ).to(device)

def create_agcrn_model(config, use_filter=False, use_time_emb=False, use_nodes=False):
    return AGCRN(
        config=config,
        num_nodes=num_nodes, 
        input_dim=1, 
        rnn_units=64, 
        output_dim=1, 
        seq_len=12, 
        horizon=12, 
        num_layers=2, 
        default_graph=True, 
        embed_dim=10, 
        cheb_k=2,
        use_nodes=use_nodes,
        use_filter=use_filter,
        use_time_emb=use_time_emb,
        residual_channels=32,
        dropout_rate=0.2,
    ).to(device)


Using device: cuda


In [23]:
configs = {}

flags = ['use_filter', 'use_nodes', 'use_time_emb']
combinations = list(itertools.product([False, True], repeat=3))

combinations = [
    (False, False, False),
    (False, False, True),
    (False, True, False),
    (False, True, True),
    (True, True, True),
]

i = 1
all_i = len(combinations)

In [145]:
for combo in combinations:
    config = dict(zip(flags, combo))
    print(f"===({i}/{all_i}) Запуск конфигурации: {config} ===")

    # Создание модели
    model = create_gwnet_model(configs, **config)

    # Оптимизатор и функция потерь
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # TensorBoard логирование
    log_dir = "runs/GWNet/PEMS08 LAST/" + "_".join([f"{k[4:]}={int(v)}" for k, v in config.items()])
    writer = SummaryWriter(log_dir=log_dir)

    # Тестовый вывод параметров модели
    train_iter = iter(train_loader)
    history_data = next(train_iter)[0].to(device)
    summary(model, input_data=history_data)

    # Обучение и валидация
    train_val_test_model(model, optimizer, criterion, train_loader, val_loader, test_loader,
                         epochs=50, writer=writer)
    i += 1

===(1/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': False, 'use_time_emb': False} ===


===(2/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': False, 'use_time_emb': True} ===


===(3/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': True, 'use_time_emb': False} ===


===(4/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': True, 'use_time_emb': True} ===


===(5/5) Запуск конфигурации: {'use_filter': True, 'use_nodes': True, 'use_time_emb': True} ===


In [24]:
i=1
for combo in combinations:
    config = dict(zip(flags, combo))
    print(f"===({i}/{all_i}) Запуск конфигурации: {config} ===")

    # Создание модели
    model = create_agcrn_model(configs, **config)

    # Оптимизатор и функция потерь
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # TensorBoard логирование
    log_dir = "runs/AGCRN/METR-LA LAST/" + "_".join([f"{k}={int(v)}" for k, v in config.items()])
    writer = SummaryWriter(log_dir=log_dir)

    # Тестовый вывод параметров модели
    train_iter = iter(train_loader)
    history_data = next(train_iter)[0].to(device)
    summary(model, input_data=history_data)

    # Обучение и валидация
    train_val_test_model(model, optimizer, criterion, train_loader, val_loader, test_loader,
                         epochs=50, writer=writer)
    i += 1

===(1/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': False, 'use_time_emb': False} ===


===(2/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': False, 'use_time_emb': True} ===


===(3/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': True, 'use_time_emb': False} ===


===(4/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': True, 'use_time_emb': True} ===


===(5/5) Запуск конфигурации: {'use_filter': True, 'use_nodes': True, 'use_time_emb': True} ===


In [10]:
i=1

for combo in combinations:
    config = dict(zip(flags, combo))
    print(f"===({i}/{all_i}) Запуск конфигурации: {config} ===")

    # Создание модели
    model = create_stgcn_model(configs, gso=adj_tensor, **config)

    # Оптимизатор и функция потерь
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # TensorBoard логирование
    log_dir = "runs/STGCN/METR-LA LAST/" + "_".join([f"{k}={int(v)}" for k, v in config.items()])
    writer = SummaryWriter(log_dir=log_dir)

    # Тестовый вывод параметров модели
    train_iter = iter(train_loader)
    history_data = next(train_iter)[0].to(device)
    summary(model, input_data=history_data)

    # Обучение и валидация
    train_val_test_model(model, optimizer, criterion, train_loader, val_loader, test_loader,
                         epochs=50, writer=writer)
    i+=1

===(1/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': False, 'use_time_emb': False} ===


===(2/5) Запуск конфигурации: {'use_filter': False, 'use_nodes': False, 'use_time_emb': True} ===


TypeError: UniversalInputAdapter.__init__() got an unexpected keyword argument 'use_filter_gate'